# BytePlus Registration - v3 (requests only, no Playwright)

Run all cells in order. Takes ~2 minutes.

In [ ]:
# Cell 1: Check IP (must be Google/non-blocked IP)
import requests
ip_info = requests.get('https://ipinfo.io/json', timeout=10).json()
print('IP:', ip_info.get('ip'))
print('Org:', ip_info.get('org'))
print('Country:', ip_info.get('country'))
print('City:', ip_info.get('city'))

In [ ]:
# Cell 2: Full registration (pure requests, no browser)
import requests, re, time, random, string, json

def gen_rand(n): return ''.join(random.choices(string.ascii_lowercase + string.digits, k=n))

CSRF = '8b13c7ddf17a645320fff792b453c01c'
HDRS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36',
    'Accept': 'application/json, text/plain, */*',
    'Accept-Language': 'en-US,en;q=0.9',
    'Content-Type': 'application/json',
    'Origin': 'https://console.byteplus.com',
    'Referer': 'https://console.byteplus.com/auth/signup',
    'sec-ch-ua': '"Google Chrome";v="131", "Chromium";v="131", "Not_A Brand";v="24"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'X-Csrf-Token': CSRF,
}

ARK = 'https://ark.cn-beijing.volces.com/api/v3'
MODEL = 'doubao-seedance-2-5-260628'

def test_key(k):
    r = requests.post(f'{ARK}/contents/generations/tasks',
        headers={'Authorization': f'Bearer {k}', 'Content-Type': 'application/json'},
        json={'model': MODEL, 'content': [{'type': 'text', 'text': 'cat running'}]},
        timeout=14)
    ec = ''
    try: ec = r.json().get('error', {}).get('code', '')
    except: pass
    return r.status_code, ec

# Get guerrillamail email
gm = requests.Session()
gdata = gm.get('https://api.guerrillamail.com/ajax.php?f=get_email_address',
               headers={'User-Agent': 'Mozilla/5.0'}, timeout=10).json()
email = gdata['email_addr']; sid = gdata['sid_token']
username = 'bpdev' + gen_rand(8)
PASSWORD = 'BpP1us!2026#A'
print(f'Email: {email}')
print(f'Username: {username}')

# Get session
sess = requests.Session()
sess.headers.update(HDRS)
sess.get('https://console.byteplus.com/auth/signup', timeout=15)
csrf = sess.cookies.get('csrfToken', CSRF)
print(f'CSRF: {csrf[:20]}')

# Check email + username
r1 = sess.post('https://console.byteplus.com/api/passport/account/checkEmailUniqueV2',
    headers={'Content-Type': 'application/json', 'X-Csrf-Token': csrf},
    json={'Email': email}, timeout=10)
print(f'Email check: {r1.status_code}')

r2 = sess.post('https://console.byteplus.com/api/passport/account/nameCheck',
    headers={'Content-Type': 'application/json', 'X-Csrf-Token': csrf},
    json={'Identity': username}, timeout=10)
print(f'Username check: {r2.status_code}')

# Send OTP
otp_r = sess.post('https://console.byteplus.com/api/passport/login/sendSignupEmailforBP',
    headers={'Content-Type': 'application/json', 'X-Csrf-Token': csrf},
    json={'Email': email}, timeout=15)
print(f'OTP send: {otp_r.status_code} {otp_r.text[:100]}')

# Wait for OTP
otp = None
for i in range(30):
    time.sleep(3)
    gm2 = gm.get(f'https://api.guerrillamail.com/ajax.php?f=check_email&seq=0&sid_token={sid}',
                 headers={'User-Agent': 'Mozilla/5.0'}, timeout=10).json()
    for mail in gm2.get('list', []):
        if any(x in mail.get('mail_subject','').lower() for x in ['byteplus','verify','code']):
            br = gm.get(f'https://api.guerrillamail.com/ajax.php?f=fetch_email&email_id={mail["mail_id"]}&sid_token={sid}',
                       headers={'User-Agent': 'Mozilla/5.0'}, timeout=10).json()
            body_text = re.sub(r'<[^>]+>', ' ', br.get('mail_body','') + br.get('mail_body_html',''))
            codes = re.findall(r'\b(\d{6})\b', body_text)
            if codes: otp = codes[0]; break
    if otp: print(f'OTP: {otp}'); break
    print(f'  Checking {(i+1)*3}s...')

if not otp:
    print('NO OTP RECEIVED!')
else:
    print(f'Got OTP: {otp}')
    
    # Register via /signup endpoint (bypasses shark URL)
    reg_body = {
        'Email': email, 'Identity': username, 'Password': PASSWORD,
        'OptIn': False, 'Code': otp,
        'f_console_sign_up_url_source': 'https://console.byteplus.com/auth/signup',
        'Utm': {}, 'Locale': 'en', 'apiVersion': 'v2',
        'captcha_code': 'bypass',
    }
    
    # Try /signup endpoint
    r_reg = requests.post(
        'https://console.byteplus.com/api/passport/login/signup',
        headers=HDRS,
        json=reg_body,
        timeout=15)
    ec = ''
    try: ec = r_reg.json().get('ResponseMetadata', {}).get('Error', {}).get('Code', '')
    except: pass
    print(f'Register /signup: {r_reg.status_code} ec={ec!r}')
    print(f'Response: {r_reg.text[:500]}')
    
    if r_reg.status_code == 200 and not ec:
        print('\n=== REGISTRATION SUCCESSFUL ===')
        print(f'Email: {email}')
        print(f'Username: {username}')
        print(f'Password: {PASSWORD}')
        print('\nNext: Go to https://console.byteplus.com/ark/region:ark+ap-southeast-1/apiKey')
        print('Create API key and test it!')
    elif ec == 'ErrorNeedCaptcha':
        print('\nNeed captcha - Google IP triggered captcha?')
        print('Try: captcha_code = bypass OR solve captcha')
    elif ec == 'OperationTooFrequent':
        print('\nRate limited - this IP has been used before!')
        print('Need a fresh IP that has never touched BytePlus')
    else:
        print(f'\nUnexpected result: ec={ec!r}')

In [ ]:
# Cell 3: If registration succeeded - get API key
# (Run this ONLY after Cell 2 shows success)

# Manual step: login to console and create API key
# 1. Go to: https://console.byteplus.com
# 2. Login with email/password from Cell 2
# 3. Go to: https://console.byteplus.com/ark/region:ark+ap-southeast-1/openManagement
# 4. Enable doubao-seedance-2-5-260628
# 5. Go to: https://console.byteplus.com/ark/region:ark+ap-southeast-1/apiKey
# 6. Create API key
# 7. Test with: python test below

# Test an API key
API_KEY = 'PASTE_YOUR_KEY_HERE'

if API_KEY != 'PASTE_YOUR_KEY_HERE':
    ARK = 'https://ark.cn-beijing.volces.com/api/v3'
    MODEL = 'doubao-seedance-2-5-260628'
    r = requests.post(f'{ARK}/contents/generations/tasks',
        headers={'Authorization': f'Bearer {API_KEY}', 'Content-Type': 'application/json'},
        json={'model': MODEL, 'content': [{'type': 'text', 'text': 'cat running in garden'}]},
        timeout=14)
    ec = ''
    try: ec = r.json().get('error', {}).get('code', '')
    except: pass
    print(f'Key test: {r.status_code} ec={ec!r}')
    if not ec:
        print('KEY WORKS! Add to gateway.')
    elif ec == 'ModelNotOpen':
        print('Key valid but Seedance 2.5 not activated. Enable in ARK console.')